<img src="https://github.com/moroneyt/MXB301/raw/main/resources/qutlogo.jpg">

# MXB301 Mathematics of AI
# Lesson 7: Convolutions and Image Classification

#### Tim Moroney, 2026

A lesson where we introduce convolutions and build a classifier for the MNIST data set.

# Package management
We start by installing the required packages. If you're running on Colab this will just download pre-compiled code. Otherwise be prepared to wait a few minutes for installation the first time you run. Feel free to read ahead while you wait.

In [ ]:
import Pkg
if haskey(ENV, "COLAB_GPU") # check if we're on Colab
  if !isfile("/content/MXB301_2026_01_CPU.tgz") # check if we've already downloaded
    # download precompiled Julia environment for Colab
    run(`gdown https://drive.google.com/uc\?id=1mT9XFadzdfK8CWb5a7BYLUkd2RTi2eZc`)

    # replace Colab's Julia environment with downloaded version
    run(`rm -rf /root/.julia`)
    run(`tar -xzf MXB301_2026_01_CPU.tgz -C /root`)
  end
else
  # For any other machine we install the packages in the usual way
  Pkg.activate(".")
  Pkg.add(["CairoMakie", "CodecZlib", "ColorSchemes", "ComponentArrays", "CondaPkg",
           "DifferentiationInterface", "Distributions", "Downloads", "FiniteDiff", "ForwardDiff",
           "HTTP", "JLD2", "LaTeXStrings", "LinearAlgebra", "Lux", "MKL", "MLUtils", "NNlib",
           "NLSolversBase", "OneHotArrays", "Optim", "PythonCall", "QuadGK", "Random",
           "SpecialFunctions", "Statistics", "StatsBase", "ToeplitzMatrices", "Zygote"])
end

using CairoMakie
using DifferentiationInterface
using LaTeXStrings
using LinearAlgebra
using Lux
using Random
using SpecialFunctions
using Statistics

using ComponentArrays: ComponentVector
using Distributions: Normal, Exponential
using Downloads: download
using ForwardDiff: Dual, partials
using JLD2: jldopen
using MLUtils: DataLoader, rand_like, randn_like
using NLSolversBase: only_fg
using NNlib: softmax, sigmoid, scatter as scattergrad, conv, ∇conv_filter, ∇conv_data, DenseConvDims
using OneHotArrays: onehot, onehotbatch, onecold
using StatsBase: crossentropy, sample, Weights
using ToeplitzMatrices: Toeplitz, Hankel
using QuadGK: quadgk

import CodecZlib
import ColorSchemes
import FiniteDiff
import HTTP
import MKL
import Optim
import Zygote

# Set the random seed for reproducibility
rng = Random.seed!(0)

# SVG format scales properly in web pages and PDFs
CairoMakie.activate!(type = "svg")

# From text to images

Our study of AI models to date has focused on language models.  But of course generative _image_ models also occupy a large part of the AI landscape.  The remainder of the unit will be devoted to covering the new mathematics required to build generative models for images.  Today's focus is the **convolution**, which is a key operation in most AI image models, precisely because of the way that images are different from language.

# Introduction to convolutions

We start with a one-dimensional presentation.  Although images are obviously two dimensional, the idea of convolutions is also relevant to signal processing (e.g. for audio applications).  We won't detour down the path of one-dimensional AI applications though, so the presentation here is just a mathematical stepping stone to 2D.

Imagine you have a one-dimensional array of values $x$, and you want to compute a kind of smoothed average of the values in $x$ based on the neighbouring values.  To be concrete, call the resulting array $y$, then $y_i$ might satisfy, for example
$$
y_i = 0.1 x_{i-2} + 0.2 x_{i-1} + 0.4 x_i + 0.2 x_{i+1} + 0.1 x_{i+2}.
$$

So this is computing a weighted average of each value in $x$ and its four neighbouring values (two either side).  The particular weights we have chosen here are
$$
w = [0.1, 0.2, 0.4, 0.2, 0.1]\,.
$$

We will call the array $w$ the _filter_.  A calculation like this might be useful for smoothing, or "filtering" out noise in a signal, for example.  Each value $x_i$ might be a noisy measurement, so by taking the weighted average of a few neighbouring values we keep the same trend in the signal, but smooth out some of the noise.

Conceptually this is the right idea, but in practice it will turn out to be more convenient if we shift the indices in the sum and instead use
$$
y_i = 0.1 x_i + 0.2 x_{i+1} + 0.4 x_{i+2} + 0.2 x_{i+3} + 0.1 x_{i+4}.
$$

It's the same calculation of course, just shifting our perspective on the indexing to avoid subtraction.

The calculation we are performing is then
$$
y_i = \sum_{i' = 1}^k w_{i'}\ x_{i+i'-1}
$$



## Noisy signal example

Here's a small example.  We build an array of $n = 16$ linearly increasing signal values, contaminated by noise.  The increasing trend is visible in the plot, but it's somewhat obscured by the noise.

In [ ]:
n = 16
x = (1:n) + randn(n)       # noisy signal
fig, ax = lines(1:n, 1:n, label="linear", linestyle=:dash, color=:black)
lines!(1:n, x, label="noisy signal", linewidth=2)
axislegend(ax, position=:lt)
fig

## Smoothing with a filter

Now we will apply the smoothing effect using our weighted average filter.

Because we are averaging over two neighbours left and right, the resulting signal $y$ will be have fewer elements than $x$.  If the size of the filter is $k$, then the length of $y$ will be $m = n - k + 1$.

In [ ]:
w = [0.1,0.2,0.4,0.2,0.1]  # filter weights
@show n = length(x)
@show k = length(w)
@show m = n - k + 1;

## Applying the filter

The sum we want,
$$
y_i = \sum_{i' = 1}^k w_{i'}\ x_{i+i'-1},\quad i = 1,\ldots, m
$$
could be calculated with a pair of loops

In [ ]:
y = zeros(m)
for i = 1:m
    for i′ = 1:k
        y[i] += w[i′] * x[i+i′-1]
    end
end
y

## As a comprehension
Alternatively, we can build the sum with a one-liner using a piece of syntax called an  _array comprehension_.  The loops are still there, just written more compactly in a notation that more closely matches the mathematics.

$$
y_i = \sum_{i' = 1}^k w_{i'}\ x_{i+i'-1},\quad i = 1,\ldots, m
$$


In [ ]:
y = [sum(w[i′] * x[i+i′-1] for i′ = 1:k) for i = 1:m]

## Plot the smoothed signal
We can plot this smoothed version of the signal to compare it with the original.  The new signal has fewer entries but more closely captures the linear trend, owing to the random noise being averaged out to some extent.

In [ ]:
lines!(1+k÷2 : n-k÷2, y, label="smoothed", linewidth=2)
axislegend(ax, position=:lt)
fig

##
Now this is just one example of a convolution, with a clear visual interpretation.  But in general, the filter weights $w$ could be anything at all -- not necessarily positive, not necessarily summing to 1, not necessarily left-to-right symmetric; just some values.  And the calculation performed,
$$
y_i = \sum_{i' = 1}^k w_{i'}\ x_{i+i'-1},\quad i = 1,\ldots, m
$$
is just a linear transformation that maps a vector $x \in \mathbb{R}^n$ to a vector $y \in \mathbb{R}^m$.

Speaking of linear transformations, you know from linear algebra that any linear transformation can be written as a matrix-vector product.  So what is the matrix $W \in \mathbb{R}^{m \times n}$ such that $y = W x$ represents the convolution of $x$ with the filter $w$?

Looking at the formula above, the generic row of $W$ must comprise the filter weights with zeros either side:
$$
W_\textrm{row} = [0, \ldots, 0, w_1, w_2, \ldots, w_k, 0, 0, \ldots, 0]
$$
so that the dot product $W_\textrm{row} \cdot x$ is precisely the required sum.

##
Hence, the full matrix $W$ has this same structure for each row, with the filter weights sliding across one position with each row -- a kind of "sliding window" if you like.

The first row has the filter weights as entries 1--5.  The second row has the filter weights as entries 2--6.  And so on.

In [ ]:
W = Toeplitz([w[1]; zeros(m-1)], [w; zeros(m-1)])  # Toeplitz to be explained shortly

##
Sure enough, the calculation $y = W x$ produces the correct result.

In [ ]:
y = W * x

## Toeplitz matrices
You will have noticed we used the `Toeplitz` matrix type to build $W$.  A [Toeplitz matrix](https://en.wikipedia.org/wiki/Toeplitz_matrix) is one in which each diagonal (descending from left to right) is comprised of a single value.  Looking at $W$ above you can see this in action: there is a $0.1$ diagonal, a $0.2$ diagonal, a $0.4$ diagonal, and so on.

So a 1D convolution is elegantly understood in terms of a linear transformation using a Toeplitz matrix:
$$
y = W x\,.
$$

This is good, because we plan to use convolutions in neural nets where the filter weights are _learned_, so we're going to need to propagate gradients through this operation.  And now that we have written the convolution operation as a matrix product, matrix calculus immediately reveals how to propagate the gradient of this operation with respect to the _data_ $x$:
$$
\nabla_x L = W^\top\, \nabla_y L\,.
$$


##
For example, just choosing a random $\nabla_y L$ for demonstration, we would compute $∇_xL$ as


In [ ]:
# Suppose this ∇y comes from the previous layer in back propagation
∇y = randn(m)

# Here's how we propagate through the convolution
∇x = W' * ∇y

##
But we're not completely done with this story.  In our back propagation algorithm we will also crucially require the gradient with respect to the _filter_ $w$, $\nabla_w L$.  How can we extract this gradient from the linear transformation formula $y = W x$?

We can't! Instead, we need to reformulate the convolution as a linear transformation of $w$, rather than $x$.  This too is fine though.  We just need a matrix $X$ where each $X_\textrm{row}$ is a sliding window of the data $x$, which we dot against the filter as $X_\textrm{row} \cdot w$.

As you can see, each row of $X$ is indeed a window of $x$ of length $k$, siding along by one position with each row.  But whereas $W$ was a Toeplitz matrix, the matrix $X$ is called a [Hankel matrix](https://https://en.wikipedia.org/wiki/Hankel_matrix) -- it has constant diagonals ascending from left to right.

In [ ]:
X = Hankel(x, m, k)

##
With this definition, we now have an equivalent way to formulate the convolution operation as a linear transformation of $w$ (rather than of $x$):
$$
y = X w
$$
which agrees with our earlier result.

In [ ]:
y = X * w

##
This now also reveals the correct formula for propagating the gradient with respect to the filter $w$:

$$
\nabla_w L = X^\top\ \nabla_y L
$$

In [ ]:
∇w = X' * ∇y

# Convolution vs cross-correlation
We now pause our calcuations for a moment, to discuss an important convention concerning the precise mathematical definition of a convolution.  If you look up the [definition of a convolution](https://en.wikipedia.org/wiki/Convolution#Discrete_convolution), you will typically see the convolution of two _sequences_ $f$ and $g$ notated as $f * g$, and defined as
$$
(f * g)_i = \sum_{i' = -\infty}^{\infty} f_{i'}\, g_{i - i'}
$$
Compared to our definition above, which you recall was
$$
y_i = \sum_{i' = 1}^k w_{i'}\ x_{i+i'-1}
$$
we can observe several differences.  First, if $f$ and $g$ are sequences, they have infinitely many entries.  In which case, you might as well make the sum go from $-\infty$ to $\infty$ rather than $1$ to $\infty$.  That removes the need for the "-1" correction in the indexing of $x$ which we have in our formulation.

But even accounting for this, there is another big difference in the indexing: the textbook definition of $f * g$ uses the index $i-i'$ for $g,$ whereas our definition uses $i + i'$.  There is a good reason for mathematics books to choose the $i-i'$ convention: it makes the convolution operator _commutative_.  In other words:
$$
(f * g)_i = \sum_{i' = -\infty}^{\infty} f_{i'}\, g_{i - i'} = \sum_{i' = -\infty}^{\infty} f_{i-i'}\, g_{i'} = (g * f)_i
$$

This is such a nice mathematical property to have, it makes perfect sense to define a convolution so that it has this property.  With this interpretation, neither $f$ nor $g$ is necessarily the "data" or the "filter" -- they are just two sequences being convolved, both on equal footing.

## Convention in AI

Back to our AI application though, we don't have infinite entries in the arrays, and we most definitely have a distinction between the data $x$ and the filter $w$. This justifies our choice to define convolution in the intuitive way as we did originally, without the commutativity.  In the mathematical literature, what we defined would actually be called a **cross-correlation** rather than a convolution.

But in AI and related literature, it's often (not always) just called a convolution.  And nobody really cares if the filter weights are reversed compared to the pure mathematical definition, because the weights are going to be learned from data anyway.  So in our context it really makes no difference at all, and we won't be picky about the terminology.

# Deep learning routines

Because the convolution and its gradient are so important in many neural network architectures, deep learning libraries provide highly-optimised routines for computing them.  Here we show how to compute the same convolution and gradients using the library routines.

## Reshaping

First though we need to reshape our inputs to have _three_ dimensions rather than one: $x$ needs to be of size $n \times 1 \times 1$.  The last dimension is the batch dimension, as we are used to.  The second-last dimension is the _channel_ dimension.  Channel dimensions are probably most easily understood in the context of image processing, where you may be familiar with the RGB channels of a colour image.  But even in this one-dimensional signal application, you could imagine a use for the channel dimension if $x$ were allowed to be a  _complex_-valued signal, in which case the first channel $x[:,1,:]$ would represent the real part of $x$, while the second channel $x[:,2,:]$ would represent the imaginary part of $x$.

Whatever the use for the channel dimension may be, the library routines always assume it is present, along with the batch dimension.  So we must ensure that $x$ is of size  $n \times \textrm{channels} \times \textrm{batch}$, even if $\textrm{channels}$ and $\textrm{batch}$ are singleton dimensions.  Because these dimensions are always assumed to be there, we won't "upgrade" the notation for $x$ to use a capital $X$ or anything.  It's still conceptually a one-dimensional signal.

In [ ]:
x = reshape(x, n, 1, 1)

##
The same applies for $w$.

In [ ]:
w = reshape(w, k, 1, 1)

## Convolution options
Next we set up the options we want for our convolutions.

You can see from the output that the dimensions of the arrays involved agree with those we inferred above ourselves.  You can also see that other options concerning stride, padding and dilation have been given default values.  There are many variations on the basic convolution as we have defined it, where these aspects can be varied.  A good reference to learn more about this is Chapter 9 of the [Deep Learning Book](https://www.deeplearningbook.org/contents/convnets.html).

In [ ]:
# flipkernel = "cross correlation" consistent with our definition
cdims = DenseConvDims(x, w; flipkernel=true)

## Computing convolution
We can now compute the convolution $y$ as

In [ ]:
 y = conv(x, w, cdims)

##
and the gradients $∇_xL$

In [ ]:
∇y = reshape(∇y, :, 1, 1)  # add singleton channel and batch dimensions
∇x = ∇conv_data(∇y, w, cdims)

##
and $∇_wL$, which all agree with our earlier calculations.



In [ ]:
∇w = ∇conv_filter(x, ∇y, cdims)

##
In summary, 1D convolutions can be completely understood, analysed and implemented using Toeplitz matrices and Hankel matrices, for both forward and backward passes.  But in practice, for maximum efficiency, we should use the library routines `conv`, `∇conv_data` and `∇conv_filter`.


# Convolutions in 2D
What about convolutions in 2D?  We are interested in images, after all.  Suppose we have a greyscale image $I \in \mathbb{R}^{n \times n}$.  Rather than fill it with random values, we will make a little T shape out of just zeros and ones.

In [ ]:
I = zeros(n,n)
I[n÷2:n÷2+1, 1:n÷2] .= 1
I[:, n÷2:n÷2+1] .= 1
image(I, axis=(aspect=1,))

#
We also have a two-dimensional filter $F \in \mathbb{R}^{k \times k}$.  Again, just to make this example concrete we will fill it with particular values, which happen to be designed to [detect edges](https://en.wikipedia.org/wiki/Sobel_operator).  But we won't be dwelling on the details of the filter -- our real goal is to let the AI system learn what the most appropriate filters are, not to design them ourselves.

When visualising the filter as an image, we'll use a colour map that moves from red (positive) through white (zero) to blue (negative).

In [ ]:
F = kron([-1.0,-2.0,0.0,2.0,1.0], [1.0,4.0,6.0,4.0,1.0]')
display(image(F, interpolate = false, colormap = [:red, :white, :blue], axis = (aspect=1,)))
F

#
Then the two-dimensional convolution is defined by
$$
Y_{i,j} = \sum_{i'=1}^k \sum_{j'=1}^k F_{i',\, j'}\ I_{i+i'-1,\, j+j'-1}
$$
(again, this would be called a "cross-correlation" in more mathematical literature).

Here it is as a one-liner.  The resulting filtered image has "detected" both vertical edges that were present in the original image: the rising edge (dark to light in the original image) and the falling edge (light to dark in the original image).

In [ ]:
Y = [sum(F[i′,j′] * I[i+i′-1, j+j′-1] for i′=1:k, j′=1:k) for i=1:m, j=1:m]
image(Y, axis=(aspect=1,))

#
You can appreciate how detecting the locations of vertical edges could be quite useful in classifying images.  But so would detecting the locations of horizontal edges.  Or what about edges aligned at other angles?  Or what about the presence of [specific frequency components](https://en.wikipedia.org/wiki/Gabor_filter)?  Or other features that don't even have a simple intuitive explanation?

If you allow the filter weights to be _learned_ from training data, then the convolution can be used for quite general **feature detection**.  That is the application we have in mind.  We will treat the filter weights as learnable parameters in a **convolutional layer** of the network, the same as we did for our embedding layer and dense layers in previous lessons.

## Deep learning routines

As we did with 1D convolutions, if we want to use the deep learning routines to compute convolutions and gradients thereof in 2D, we must ensure the inputs have channel and batch dimensions, and set the convolution options.

In [ ]:
# add singleton channel and batch dimensions
I = reshape(I, n, n, 1, 1)
F = reshape(F, k, k, 1, 1)

# flipkernel = "cross correlation" consistent with our definition
cdims = DenseConvDims(I[:,:,:,:], F[:,:,:,:]; flipkernel=true)

##
Then the 2D convolution is computed simply using `conv` as before.

In [ ]:
Y = conv(I, F, cdims)
image(Y[:,:,1,1,], axis = (aspect=1,))

#
The gradients can be computed similarly using `∇conv_data` and `∇conv_filter`.  To demonstrate we'll just propagate a random gradient.

Here's the gradient with respect to the data.

In [ ]:
∇Y = randn(m,m,1,1);           # random gradient to propagate

∇I = ∇conv_data(∇Y, F, cdims)  # gradient with respect to data

#
and here's the gradient with respect to the filter.

In [ ]:
∇F = ∇conv_filter(I, ∇Y, cdims)  # gradient with respect to filter

#
Be thankful you don't need to write the code to compute these gradients yourself!  It's still true that the 2D convolution is a linear operation, so we know it _could_ be represented by a matrix-vector product.  But that would require reshaping the images into vectors, and then building an enormous block Toeplitz matrix.  That's dumb.  But the actual efficient implementations are quite fiddly, and in fact there are multiple ways you could do it, and which approach ends up being fastest can depend on many little details.  We'll gladly leave all that complexity to the experts who wrote the library routines.

# The MNIST data set

Now that we understand convolutions, let's put them to good use.  We will solve a classic problem in AI, which seems very trivial by today's standards but was one of the first real successes in the field: classifying hand-written digits.  

Back when people still posted hand-written letters to each other, it was desirable that the postcode on the envelope could be automatically read, so that mail could be quickly sorted.  And so the [MNIST](https://en.wikipedia.org/wiki/MNIST) dataset was born: a collection of 70,000 $28 \times 28$ greyscale images of handwritten digits, along with their correct labels.  Let's load the data set now.

In [ ]:
# Load the MNIST dataset
url = "https://github.com/moroneyt/MXB301/raw/main/resources/MNIST.jld2"
MNISTfile = jldopen(download(url))

#
You can see from the summary below that the size of the `images` array matches our expectations: $28 \times 28 \times 1 \times 70000$.  So the channel dimension is 1 (greyscale images), and the batch dimension is 70,000.

We also see they are stored as `UInt8` data type (so, values $0-255$). It will be convenient to convert these integer values to floating point values between $0$ and $1$, so let's do that now.

In [ ]:
@show summary(MNISTfile["images"])
images = MNISTfile["images"] / 255;  # convert to 0..1 range

#
We can take a look at the first few images in the set.  Just for fun let's concatenate four of them to form a plausible postcode.

In [ ]:
# a plausible postcode
image(reduce(vcat, images[:,:,1,j] for j = 1:4))

#
These are not pure black and white images.  They are true greyscale images, with pixel values ranging from zero to one.  In particular the edges of the digits are shades of grey rather than pure black or white.

In [ ]:
images[:,:,1,1]  # here is the first image

#
The digits also come with labels, which are essential when we come to train our neural net: these are the **ground truth**, against which we can compare our model's predictions.

We see the first four labels are indeed 5041 in agreement with the image.

In [ ]:
labels = MNISTfile["labels"]

# Convolutional neural networks

Convolutional neural networks, or CNNs, are neural networks in which there is at least one convolution layer.  CNNs are ubiquitous in computer vision applications -- detecting and classifying objects in images and videos.

Why do we need convolution layers to perform this kind of task?  Well in theory we don't really _need_ them.  A sufficiently deep neural net made up of just dense layers is capable of this task in theory -- remember the [universal approximation theorem](https://en.wikipedia.org/wiki/Universal_approximation_theorem).

But by using convolution layers rather than fully connected dense layers, we are essentially baking in to the neural net architecture certain assumptions about the problem at hand.  Namely, that a feature (edge, corner, whatever) is meaningful wherever it occurs, so the same "feature detector" should be applied unchanged throughout the image.

More technically, convolutions are **translation equivariant**: if $\mathcal{L}$ represents a spatial translation, then
$$
\textrm{conv}(\mathcal{L} X) = \mathcal{L}\, \textrm{conv}(X)
$$
for any input $X$ (exercises!).

So convolution layers automatically enforce translational equivariance, whereas a fully connected dense layer would not, in general.  In fact, if a fully connected dense layer "wanted" to learn some kind of edge detection filter, for example, it would essentially have to learn to build the block Toeplitz representation of the convolution operator as its weight matrix.  By contrast a convolution layer encodes the same information in just $k^2$ weights.

Still, the actual values of the $k^2$ weights are free to be learned, so if the network "wants" to learn an edge detection filter, it can do so.  If it wants to learn some sort of [Gabor filter](https://en.wikipedia.org/wiki/Gabor_filter) it can do so.  And in fact, in any given conv layer, there will generally be more than one filter that is learned: this is how the channel dimension gets filled out.  There could be, say, 8 output channels, each of which corresponds to the convolution of the input with one of the 8 different learned filters in the layer.

# Model
OK, it's time to decide on the network architecture.  We will keep it as simple as possible.  Here's the forward pass code, which you can see comprises a:
1. conv layer (with bias)
2. nonlinear activation
3. flattening
4. dense layer
5. softmax output.

In [ ]:
# Forward pass: predict the image label
function forward(X, p)

    # Dimension bookkeeping helper for conv
    cdims = DenseConvDims(X, p.F)

    Z1 = conv(X, p.F, cdims) .+ p.bc      # 1. conv layer
    H1 = tanh.(Z1)                        # 2. activation
    V1 = reshape(H1, :, size(H1,4))       # 3. flatten
    Z2 = p.W * V1 .+ p.bd                 # 4. dense layer
    Ŷ = softmax(Z2)                       # 5. softmax for probabilities

    # Return the output along with a cache of intermediate values
    cache = (; cdims, X, Z1, H1, V1, Z2)
    return Ŷ, cache
end

# Hyper-parameters
Let's now define the model hyper-parameters.

We've chosen to learn 8 filters of size $5 \times 5$ in the conv layer (`channels_out = 8`).  Since the input image size for MNIST is $28 \times 28$, that means the output of this conv layer will be of size $24 \times 24$, i.e. $m = n - k + 1 = 28 - 5 + 1 = 24$.  This is captured in the variable `feature_map_size`.

We call the output the "feature map", since (hopefully) the network will have learned suitable filters that allow it to pick out useful features of the image (such as edges, perhaps).  The subsequent dense layer then operates on this feature map rather than operating on pure pixel value inputs.

In [ ]:
# Define model hyper-parameters
num_classes = 10                                        # digits 0--9
input_size = (28,28)                                    # MNIST image size
channels_in = 1                                         # greyscale input
filter_size = (5,5)                                     # size of conv filter
feature_map_size = input_size .- filter_size .+ 1       # size after convolution
channels_out = 8                                        # number of feature maps to generate
flattened_size = prod(feature_map_size) * channels_out  # input size to dense layer

# Weight initialisation

We will continue to use Glorot initialisation of the weights.  Remember the conv layer is equivalent to multiplication by a giant block Toeplitz matrix, so we can still use the same principle to initialise the weights for that layer. This is notwithstanding the fact that the actual convolutions are calculated a much, much more efficient way using the library routines.

Below we initialise the weight matrices and bias vectors of the appropriate size for each layer.

In [ ]:
# Initialise the weights and biases
glorot_std(m, n) = sqrt(2 / (m + n))
winit_dense(m, n) = glorot_std(m, n) .* randn(m, n)
winit_conv(f1, f2, ch_in, ch_out) = glorot_std(f1*f2*ch_in, f1*f2*ch_out) .* randn(f1, f2, ch_in, ch_out)

# Conv layer parameters
F = winit_conv(filter_size..., channels_in, channels_out)
bc = zeros(1, 1, channels_out)

# Dense layer parameters
W = winit_dense(num_classes, flattened_size)
bd = zeros(num_classes)

# The full vector of trainable parameters
p0 = ComponentVector(; F, bc, W, bd)
length(p0)

# Training set

The number of parameters in this model is much greater than our character level language model from previous lessons. Mostly this is a consequence of us now dealing with two-dimensional data (images), rather than one-dimensional data (text).

To speed up the training process, we won't train on the full set of images.  We'll just use, say, the first 10,000 images for our training set.

In [ ]:
nb = 10_000   # since we're on a CPU we won't train with the full set
X = images[:,:,:,1:nb]
Y = onehotbatch(labels[1:nb], 0:9)

# Manual run-through line by line

To ensure we have familiarity with the nuts and bolts of the model, let's run though the forward process line by line.

We don't want to process all 10,000 images at a time -- we will use batching to handle that.  So for now let's just run a small subset of twenty images through the model.

In [ ]:
Xsmall = X[:,:,:,1:20]
Ysmall = Y[:, 1:20]
p = copy(p0)
size(Xsmall), size(Ysmall)

#
The conv layer is using the library `conv` routine as we've seen before, with the filter `p.F` now being a parameter of the model.  It takes the $28 \times 28 \times 1$ images and produces $24 \times 24 \times 8$ feature maps.  The images are smaller, but there are now 8 channels.  It also has a bias term `p.bc` added prior to the activation.

In [ ]:
cdims = DenseConvDims(Xsmall, p.F)
Z1 = conv(Xsmall, p.F, cdims) .+ p.bc;      # 1. conv layer

summary(Z1)  # just check the size, no need to print it all out

#
The tanh activation then provides the nonlinearity to the layer.

In [ ]:
H1 = tanh.(Z1);                          # 2. activation
summary(H1)

#
To pass these feature maps on to the dense layer, they get flattened out into long vectors.

In [ ]:
V1 = reshape(H1, :, size(H1,4));         # 3. flatten
summary(V1)

#
Then follows a standard dense layer with its own weights `p.W` and bias `p.b`.  The dense layer maps the long vectors on to 10-dimensional vectors, which is the size we need for the output layer.

In [ ]:
Z2 = p.W * V1 .+ p.bd                 # 4. dense layer

#
The final step is the softmax to ensure we have probabilities.

The output of the model is thus the predicted probabilities for each possible image label 0--9.

This is by no means the only "right" way to build the network, but we will find it is more than sufficient to handle our task.

In [ ]:
Ŷ = softmax(Z2)                        # 5. softmax for probabilities

# Backward pass
The backward pass will provide us with no difficulty, since we already have everything we need to differentiate through the model.  The softmax and dense layers we are familiar with already, and we've seen above how to take gradients of conv layers using `∇conv_data` and `∇conv_filter`.  In fact, for this model, we only need the latter; there is no need to propagate gradients with respect to the _data_ through the conv layer, because there is no earlier layer that requires it.

In [ ]:
# Backward pass: compute gradient
function backward(p, Y, Ŷ; cache)

    # gradient vector to fill in
    g = zero(p)

    # unpack the cache of intermediate values from the forward pass
    (; cdims, X, Z1, H1, V1, Z2) = cache

    # Calculate the gradient of the loss with respect to the model parameters.
    # Each rule is simple enough, but take care!
    ∇Z2 = Ŷ - Y                         # 5. softmax with crossentropy loss
    g.W = ∇Z2 * V1'                     # 4. dense layer (weights)
    g.bd = sum(∇Z2; dims=2)             # 4. dense layer (bias)
    ∇V1 = p.W'* ∇Z2                     # 4. dense layer (data)
    ∇H1 = reshape(∇V1, size(H1))        # 3. (un)flatten
    ∇Z1 = (1 .- H1.^2) .* ∇H1           # 2. activation
    g.F = ∇conv_filter(X, ∇Z1, cdims)   # 1. conv layer (weights)
    g.bc = sum(∇Z1; dims=(1,2,4))       # 1. conv layer (bias)

    return g
end

# Batching
Another important change in the training process compared to our earlier work with character-level models is that we can't afford to process all images in our training set at once.  Forming all the intermediate tensors could easily exhaust the available memory.  Instead, we want to process the data in batches where the batch size is much less than 10,000.  For example, a batch size of $B = 100$ could be perfectly sensible.

So we introduce the notation $N = 10000$ for the total number of training examples, and continue to use $B$ for the batch size.

The loss function is still the sum of the _per-sample loss_ $\ell$ (e.g. $\ell = \textrm{crossentropy}$) over all the training examples:
$$
L(p) = \sum_{j=1}^{N} \ell(y^{(j)}, M_p(x^{(j)}))\,,
$$
and the gradient is a sum too:
$$
\nabla_p L = \sum_{j=1}^{N} \nabla_p \ell(y^{(j)}, M_p(x^{(j)}))\,.
$$

Precisely because these expressions are sums over the training examples, we can accumulate the loss, and its gradient, one batch at a time.

$$
L(p) = \sum_{j=1}^B \ell(y^{(j)}, M_p(x^{(j)})) + \sum_{j=B+1}^{2B} \ell(y^{(j)}, M_p(x^{(j)})) + \ldots + \sum_{j={N}-B+1}^{N} \ell(y^{(j)}, M_p(x^{(j)})) \,,
$$
and
$$
\nabla_p L(p) = \sum_{j=1}^B \nabla_p \ell(y^{(j)}, M_p(x^{(j)})) + \sum_{j=B+1}^{2B} \nabla_p \ell(y^{(j)}, M_p(x^{(j)})) + \ldots + \sum_{j={N}-B+1}^{N} \nabla_p \ell(y^{(j)}, M_p(x^{(j)})) \,.
$$

##
To implement batching in the `loss_and_gradient` function, we need to add the batching logic, including handling the last batch (which may be a smaller size than the others if the batch size doesn't divide evenly).

We've made one further change, which is to return the _average_ loss and  gradient, namely
$$
\frac{1}{N} L(p)\qquad\textrm{and}\qquad \frac{1}{N}\nabla_p L(p)\,.
$$

This constant scaling factor makes no difference to the training, but it means the printed values for the loss are comparable if we should one day use a different (e.g. larger) number of training examples $N$.

In [ ]:
function loss_and_gradient(p; X, Y, batchsize=100, grad=true)

    N = size(X, 4) # total number of training examples
    @assert N == size(Y, 2)

    loss = zero(eltype(p))  # initialise the loss to zero
    g = zero(p)             # initialise the gradient to zero

    # Batches
    for i = 1:batchsize:N

        # Pull out the next batch from the training data
        idx = i:min(i+batchsize-1, N)  # which samples belong to the batch
        Xbatch = X[:, :, :, idx]
        Ybatch = Y[:, idx]

        # Forward
        Ŷbatch, cache = forward(Xbatch, p)
        loss += crossentropy(Ybatch, Ŷbatch) # accumulate the loss

        # Backward
        if grad
            g += backward(p, Ybatch, Ŷbatch; cache) # accumulate the gradient
        end

    end

    # Scale for average loss
    if grad
        return loss/N, g/N
    else
        return loss/N
    end

end

## Gradient check: accuracy
Before proceeding, it's always a smart idea to check you have the gradients computed correctly.  Autodiff is your friend here.  We'll compute the gradient using our function, then check it with `Zygote`.

In [ ]:
# Calculating the gradient with our hand-coded function
loss, g_manual = loss_and_gradient(p0; X,Y)

# Calculating the gradient using reverse mode AD
g_auto = gradient(p->loss_and_gradient(p; X,Y, grad=false), AutoZygote(), p0)

# Compare the gradients to ensure they agree in every component
maximum(abs, g_manual - g_auto)

# Optimiser
With our model and gradients working well, it's time to train.  We'll use L-BFGS with 50 iterations.

In [ ]:
# L-BFGS algorithm and options
algorithm = Optim.LBFGS()
options   = Optim.Options(iterations = 50, show_trace = true, store_trace = true)

# Objective returns only function value and gradient, no Hessian
objective = only_fg(p->loss_and_gradient(p; X,Y))

# Run the optimiser
optresult = Optim.optimize(objective, p0, algorithm, options)

# Training trace
Here is the trace of the training process, showing how the loss reduced with each epoch.

In [ ]:
loss_trace = Optim.f_trace(optresult)
f = Figure()
lines(f[1,1], loss_trace, axis=(xlabel="epoch", ylabel="loss", title="Linear y-scale"))
lines(f[1,2], loss_trace, axis=(xlabel="epoch", ylabel="loss", yscale=log10, title="Log y-scale"))
f

# Trained parameter values
And these are the trained parameter values, albeit there's not much to learn by just looking at the values.

In [ ]:
p = Optim.minimizer(optresult)

# Trained model

We can now define our convenience function for inference that fixes the parameters to their trained values.  We can also define a simple function to test the accuracy on the training set, by checking which images it classifies correctly.  We judge an image to be classified correctly if the output label for the image is the most probable (i.e. it's the `argmax`).

The model identifies 98% of the training digits correctly.

In [ ]:
model(X) = forward(X, p)[1]  # convenience function for inference

# Test the model accuracy
function accuracy(Y, Ŷ)
    s = sum(argmax.(eachcol(Y)) .== argmax.(eachcol(Ŷ)))
    return s / size(Y,2)
end

accuracy(Y, model(X))

# Training data vs test set
The model achieves 98% accuracy on the images it was trained on.  But what about new images it hasn't seen before? If the model can't _generalise_ to new data, it's of no use.  Ideally, the training set we used was sufficiently representative of all plausible digit images, so that its performance on new images will be comparable.  So let's check.

We'll build a test set of images which is the _last_ 10,000 images in the MNIST database, none of which the model saw during training.

In [ ]:
# Test set images
Xtest = images[:,:,:,end-9999:end]

# Test set ground truth
Ytest = onehotbatch(labels[end-9999:end], 0:9)

# Run the model on the test set
Ŷtest = model(Xtest)

# Compare the predictions to the ground truth
accuracy(Ytest, Ŷtest)

#
So the accuracy on the test set is (predictably) not as good, but it's still reasonable.  It can be interesting to investigate which images it got wrong.  Are they quite ambiguous-looking, so that even a human might have trouble with them?

In [ ]:
# Which images were misclassified?
misclassified = findall(argmax.(eachcol(Ŷtest)) .!= argmax.(eachcol(Ytest)))
println("Number of misclassified test images: ", length(misclassified), " out of ", size(Ytest,2))

# Let's take a look at some of them
@assert length(misclassified) ≥ 12
f = Figure()
count = 1
for i = 1:3
    for j = 1:4
        idx = misclassified[count]
        actual = argmax(Ytest[:,idx])-1
        predicted = argmax(Ŷtest[:,idx])-1
        image(f[i,j], Xtest[:,:,1,idx], axis=(title="y = $actual, ŷ = $predicted",))
        hidedecorations!(current_axis())
        count += 1
    end
end
f

# Learned filters
Another thing we can visualise are the filters the network learned. Certainly there are hints of edge detection-looking patterns in the output. Perhaps if you stare long enough you can convince yourself you see other patterns too.  But honestly, it can be very hard to interpret what the network has actually learned. Somehow these particular patterns work well in conjunction with the following layers, so that the final output is effective at classifying digits. Ultimately, that's all that the network is trained to do -- if there happens to be some intuitive explanation to the intermediate layer weights then all the better, but there's no guarantee.

In [ ]:
f = Figure()
count = 1
for i = 1:2
    for j = 1:4
        image(f[i,j], p.F[:,:,1,count], interpolate = false, colormap = [:red, :white, :blue], axis = (aspect=1,))
        hidedecorations!(current_axis())
        count += 1
    end
end
f

# Conclusion

In this lesson, we:

* introduced the mathematical operation of convolution and related cross correlation
* showed how to interpret this operation as a matrix product, and thereby derived gradient formulas
* introduced the idea of convolutional layers as learnable filters for image based models
* wrote the forward and backward pass for a simple image classifier model
* trained the model on the MNIST handwritten digit dataset
* evaluated the model's performance using classification accuracy on unseen test images

In the next lesson we will see how to implement and train this same model using high level deep learning library components.


